# Phase 1 - Matched training: **ViT** models (ArSL, 32 classes)

Fine-tunes ImageNet-pretrained Vision-Transformer backbone(s) under the
**matched recipe** shared with the CNN notebook (same optimizer, cosine+warmup
schedule, augmentation, epochs, seeds - CLAUDE.md constraint #2). The **only**
difference between this notebook and `01a_train_cnn` is the `MODELS` list below.

**Data (leakage-free by construction):** train/val come from a stratified 80/20
split of `Mosl_alphabet_train`; the held-out **test** set is the *physically
separate* `Mosl_alphabet_test` folder, so no test image was seen in training.
Trains one run per `(model, seed)`, keeps the best-val checkpoint, and evaluates
on that test set. Outputs checkpoints / per-epoch logs / confusion matrices / a
seed-aggregated summary to `OUT_DIR`.

**Run order:** leave `SMOKE_TEST = True` for a 1-epoch sanity pass, then set it
to `False` and Run-All for the real 3-seed run.

In [ ]:
# torch/torchvision are preinstalled on Kaggle GPU images; ensure timm is present.
!pip -q install -U timm

## 1. Select models (only cell that differs CNN vs ViT)

In [ ]:
# ViT family. Primary (capacity-matched) = DeiT-Small (~22M) + Swin-Tiny (~28M).
# ViT-B/16 is a SECONDARY reference point only (~86M, not capacity-matched).
MODELS = [
    {"key": "deit_small", "timm_name": "deit_small_patch16_224", "family": "vit"},
    {"key": "swin_tiny",  "timm_name": "swin_tiny_patch4_window7_224", "family": "vit"},
    # optional secondary reference:
    # {"key": "vit_b16", "timm_name": "vit_base_patch16_224", "family": "vit"},
]

## 2. Config + matched recipe

In [ ]:
from pathlib import Path
import torch

# ---- dataset paths (Kaggle, read-only) ----------------------------------
TRAIN_ROOT = Path("/kaggle/input/datasets/youssefnouiouar1/sing-language-recognition/SLR/Mosl_alphabet_train")
TEST_ROOT  = Path("/kaggle/input/datasets/youssefnouiouar1/sing-language-recognition/SLR/Mosl_alphabet_test")
OUT_DIR    = Path("/kaggle/working/xaislr/phase1")   # checkpoints, logs, results, split record

CLASSES = ["ain", "al", "aleff", "bb", "dal", "dha", "dhad", "fa", "gaaf",
           "ghain", "ha", "haa", "jeem", "kaaf", "khaa", "la", "laam", "meem",
           "nun", "ra", "saad", "seen", "sheen", "ta", "taa", "thaa", "thal",
           "toot", "waw", "ya", "yaa", "zay"]
NUM_CLASSES  = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(sorted(CLASSES))}

# ---- train/val split (test = the separate TEST_ROOT folder) --------------
VAL_FRACTION = 0.20        # 80/20 train/val split of TRAIN_ROOT
SPLIT_SEED   = 0           # FIXED so CNN & ViT (and every model seed) share one split

# ---- matched recipe: IDENTICAL across CNN and ViT (constraint #2) --------
RECIPE = dict(
    optimizer="adamw",
    base_lr=5e-4,          # may be tuned per family; schedule SHAPE stays fixed
    weight_decay=0.05,
    warmup_epochs=3,
    epochs=40,
    label_smoothing=0.1,
    grad_clip_norm=1.0,
    amp=True,
)
SEEDS       = [0, 1, 2]    # >= 3 seeds for significance testing
IMG_SIZE    = 224
BATCH_SIZE  = 64           # lower if you hit CUDA OOM
NUM_WORKERS = 2

# ---- quick pipeline check first: 1 epoch, 1 seed, small subset ----------
SMOKE_TEST = True          # <-- set False for the real multi-seed run
if SMOKE_TEST:
    SEEDS = [0]
    RECIPE = {**RECIPE, "epochs": 1, "warmup_epochs": 0}
    SMOKE_SUBSET = 512     # cap images per split during the smoke test
else:
    SMOKE_SUBSET = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Fail early with a helpful listing if a dataset path is wrong.
for _label, _root in [("TRAIN", TRAIN_ROOT), ("TEST", TEST_ROOT)]:
    if not _root.exists():
        print(f"{_label}_ROOT not found:", _root)
        base = Path("/kaggle/input")
        if base.exists():
            print("directories under /kaggle/input:")
            for p in sorted(base.rglob("*")):
                if p.is_dir():
                    print("  ", p)
        raise FileNotFoundError(f"Fix {_label}_ROOT above.")

print("device:", DEVICE, "| classes:", NUM_CLASSES,
      "| models:", [m["key"] for m in MODELS])
print("SMOKE_TEST:", SMOKE_TEST, "| seeds:", SEEDS, "| epochs:", RECIPE["epochs"])

## 3. Data: 80/20 train/val split + separate test folder

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".tif", ".tiff", ".webp"}

def scan_folder(root, classes):
    """Scan an image folder -> DataFrame(image_path [absolute], label).

    Handles either a class-subfolder layout (root/<class>/<imgs>) or a flat
    folder of class-named files (root/<class>*.jpg, e.g. one image per class).
    """
    root = Path(root)
    class_set = set(classes)
    classes_by_len = sorted(classes, key=len, reverse=True)   # 'aleff' before 'al'
    subdirs = [d for d in root.iterdir() if d.is_dir()]
    rows = []
    if subdirs:                                   # class-subfolder layout
        for cls in classes:
            cdir = root / cls
            if not cdir.is_dir():
                continue
            for f in sorted(cdir.iterdir()):
                if f.is_file() and f.suffix.lower() in IMG_EXTS:
                    rows.append((str(f), cls))
    else:                                         # flat layout: label from filename
        for f in sorted(root.iterdir()):
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                stem = f.stem.lower()
                lbl = stem if stem in class_set else next(
                    (c for c in classes_by_len if stem.startswith(c.lower())), None)
                if lbl is not None:
                    rows.append((str(f), lbl))
    return pd.DataFrame(rows, columns=["image_path", "label"])

def build_frames():
    """Train/val = stratified 80/20 of TRAIN_ROOT; test = the separate TEST_ROOT."""
    full_train = scan_folder(TRAIN_ROOT, CLASSES)
    test_df    = scan_folder(TEST_ROOT, CLASSES)
    assert len(full_train) > 0, f"No training images under {TRAIN_ROOT}"
    assert len(test_df) > 0,    f"No test images under {TEST_ROOT}"

    train_df, val_df = train_test_split(
        full_train, test_size=VAL_FRACTION, stratify=full_train["label"],
        random_state=SPLIT_SEED)
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)

    rec = OUT_DIR / "splits"          # record the split paths for provenance
    rec.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(rec / "train.csv", index=False)
    val_df.to_csv(rec / "val.csv", index=False)
    test_df.to_csv(rec / "test.csv", index=False)
    print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  "
          f"| classes: {full_train['label'].nunique()} train / {test_df['label'].nunique()} test")
    return train_df, val_df, test_df

def build_transforms(train):
    if not train:
        resize = int(round(IMG_SIZE * 256 / 224))
        return transforms.Compose([
            transforms.Resize(resize), transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    # matched augmentation; NO horizontal flip (orientation-sensitive letters)
    return transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.0),
        transforms.RandAugment(num_ops=2, magnitude=7),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class ASLDataset(Dataset):
    def __init__(self, df, class_to_idx, transform, subset=None, seed=0):
        if subset is not None and len(df) > subset:
            df = df.sample(n=subset, random_state=seed)
        self.df = df.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row["image_path"]) as im:   # absolute path
            img = self.transform(im.convert("RGB"))
        return img, self.class_to_idx[row["label"]]

def build_loaders(frames, seed):
    train_df, val_df, test_df = frames
    specs = [("train", train_df, True), ("val", val_df, False), ("test", test_df, False)]
    loaders = {}
    for name, dfr, is_train in specs:
        ds = ASLDataset(dfr, CLASS_TO_IDX, build_transforms(is_train),
                        subset=SMOKE_SUBSET, seed=seed)
        loaders[name] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=is_train,
                                   num_workers=NUM_WORKERS, pin_memory=True,
                                   drop_last=is_train)
    return loaders

## 4. Model factory (timm, ImageNet-pretrained)

In [ ]:
import timm

def build_model(timm_name, num_classes, pretrained=True):
    """ImageNet-pretrained backbone + fresh num_classes head (timm).

    Same call for CNN and ViT so capacity/pretraining are consistent.
    """
    return timm.create_model(timm_name, pretrained=pretrained, num_classes=num_classes)

## 5. Matched recipe: seeding, optimizer, cosine+warmup, loss

In [ ]:
import math
import random
import torch.nn as nn

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True   # speed; seeds are still logged

def build_optimizer(model, base_lr, weight_decay):
    # no weight decay on norm/bias (1-D) params -- standard practice
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim == 1 or name.endswith(".bias") else decay).append(p)
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=base_lr)

def build_scheduler(optimizer, warmup_epochs, epochs):
    """Linear warmup then cosine decay (fixed schedule shape for both families)."""
    def lr_lambda(epoch):
        if warmup_epochs > 0 and epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def build_criterion(label_smoothing):
    return nn.CrossEntropyLoss(label_smoothing=label_smoothing)

## 6. Train / eval engine (AMP, grad-clip)

In [ ]:
from sklearn.metrics import confusion_matrix

# AMP helpers, compatible across torch versions.
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(enabled): return _GradScaler("cuda", enabled=enabled)
    def amp_ctx(enabled):     return _autocast("cuda", enabled=enabled)
except Exception:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(enabled): return _GradScaler(enabled=enabled)
    def amp_ctx(enabled):     return _autocast(enabled=enabled)

def train_one_epoch(model, loader, optimizer, criterion, scaler, amp, grad_clip):
    model.train()
    tot, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_ctx(amp):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        if grad_clip:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        bs = x.size(0)
        loss_sum += loss.item() * bs
        tot += bs
        correct += (out.argmax(1) == y).sum().item()
    return loss_sum / tot, correct / tot

@torch.no_grad()
def evaluate(model, loader, criterion, amp, want_confmat=False):
    model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    ys, ps = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with amp_ctx(amp):
            out = model(x)
            loss = criterion(out, y)
        bs = x.size(0)
        loss_sum += loss.item() * bs
        tot += bs
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        if want_confmat:
            ys.append(y.cpu().numpy())
            ps.append(pred.cpu().numpy())
    cm = None
    if want_confmat and ys:
        cm = confusion_matrix(np.concatenate(ys), np.concatenate(ps),
                              labels=list(range(NUM_CLASSES)))
    return loss_sum / tot, correct / tot, cm

## 7. Run: train every (model, seed)

In [ ]:
import time

for sub in ["checkpoints", "logs", "results"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

frames = build_frames()          # one fixed train/val split + the separate test set

records = []
for m in MODELS:
    for seed in SEEDS:
        tag = f"{m['key']}_seed{seed}"
        print(f"\n===== {tag} =====")
        seed_everything(seed)
        loaders = build_loaders(frames, seed)
        model = build_model(m["timm_name"], NUM_CLASSES, pretrained=True).to(DEVICE)
        optimizer = build_optimizer(model, RECIPE["base_lr"], RECIPE["weight_decay"])
        scheduler = build_scheduler(optimizer, RECIPE["warmup_epochs"], RECIPE["epochs"])
        criterion = build_criterion(RECIPE["label_smoothing"])
        amp = bool(RECIPE["amp"]) and DEVICE == "cuda"
        scaler = make_scaler(amp)

        best_val, best_state, history = -1.0, None, []   # -1 so epoch 0 always captures weights
        for epoch in range(RECIPE["epochs"]):
            t0 = time.time()
            tr_loss, tr_acc = train_one_epoch(
                model, loaders["train"], optimizer, criterion, scaler, amp,
                RECIPE["grad_clip_norm"])
            va_loss, va_acc, _ = evaluate(model, loaders["val"], criterion, amp)
            cur_lr = optimizer.param_groups[0]["lr"]
            history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                                val_loss=va_loss, val_acc=va_acc, lr=cur_lr))
            scheduler.step()
            if va_acc > best_val:
                best_val = va_acc
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            print(f"epoch {epoch:02d} | train {tr_acc:.3f} | val {va_acc:.3f} "
                  f"| best {best_val:.3f} | {time.time() - t0:.0f}s")

        if best_state is not None:
            model.load_state_dict(best_state)
        te_loss, te_acc, cm = evaluate(model, loaders["test"], criterion, amp,
                                       want_confmat=True)
        print(f"{tag} TEST acc = {te_acc:.4f}")

        torch.save({"model": m, "seed": seed, "state_dict": best_state,
                    "best_val_acc": best_val, "test_acc": te_acc,
                    "recipe": RECIPE, "classes": sorted(CLASSES)},
                   OUT_DIR / "checkpoints" / f"{tag}_best.pt")
        pd.DataFrame(history).to_csv(OUT_DIR / "logs" / f"{tag}_history.csv", index=False)
        if cm is not None:
            np.save(OUT_DIR / "results" / f"{tag}_confmat.npy", cm)
        records.append(dict(model=m["key"], family=m["family"], seed=seed,
                            best_val_acc=best_val, test_acc=te_acc))

pd.DataFrame(records).to_csv(OUT_DIR / "results" / "runs.csv", index=False)
print("\nsaved", OUT_DIR / "results" / "runs.csv")

## 8. Summary (mean +/- std over seeds)

In [ ]:
runs = pd.DataFrame(records)
summary = (runs.groupby(["family", "model"])
               .agg(test_acc_mean=("test_acc", "mean"),
                    test_acc_std=("test_acc", "std"),
                    val_acc_mean=("best_val_acc", "mean"),
                    n_seeds=("seed", "nunique"))
               .reset_index())
summary.to_csv(OUT_DIR / "results" / "summary.csv", index=False)
print(summary.to_string(index=False))
summary